# IOAI — 2026 Regional 11 12 Nurse (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-nurse.zip', '/tmp/nurse.zip')
    with zipfile.ZipFile('/tmp/nurse.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train_data.csv' if os.path.exists('data/train_data.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import roc_auc_score

In [ ]:
root_path = "data"

seed = 42

# Subtask 1

In [ ]:
df_train = pd.read_csv(f"{root_path}/train_data.csv")

In [ ]:
subtask1 = [df_train["day_28_flg"].isna().sum()]
subtask1

In [ ]:
f"{subtask1[0] / len(df_train) * 100:.2f}% missing"

# Subtask 2

In [ ]:
labeled_df = df_train.dropna(subset=["day_28_flg"])

sofa_gt10_mask = labeled_df["sofa_first"] > 10
sofa_le10_mask = labeled_df["sofa_first"] <= 10

mortality_gt10 = labeled_df[sofa_gt10_mask]["day_28_flg"].mean() * 100
mortality_le10 = labeled_df[sofa_le10_mask]["day_28_flg"].mean() * 100

In [ ]:
subtask2 = [mortality_gt10, mortality_le10]
subtask2

# Subtask 3

## Data

In [ ]:
def prep_df(df: pd.DataFrame):
    df = df.drop(["ID"], axis=1, errors='ignore')

    # df = df.dropna(axis=1)
    df = df.select_dtypes(exclude='object')

    columns = [c for c in df.columns if c != "day_28_flg"]
    return df, columns

In [ ]:
train_feats, features = prep_df(df_train)
train_feats.head()

In [ ]:
scaler = StandardScaler()

train_feats = scaler.fit_transform(train_feats[features])

In [ ]:
train_for_imp = pd.DataFrame(train_feats, columns=features)
train_for_imp["day_28_flg"] = df_train["day_28_flg"].values

imputer = KNNImputer(n_neighbors=10)
train_imputed = imputer.fit_transform(train_for_imp)

train_targets = train_imputed[:, -1]
train_targets = (train_targets > 0.5).astype(int)
train_imputed = train_imputed[:, :-1]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train_feats, train_targets, test_size=0.2, random_state=seed)

## Model

In [ ]:
def evaluate(clf):
    cv = cross_val_score(clf, X_train, y_train, cv=3, scoring="roc_auc")
    cv = cv.mean() - cv.std()

    clf.fit(X_train, y_train)
    preds = clf.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, preds), cv

In [ ]:
rf = RandomForestClassifier(n_estimators=500, max_depth=5, class_weight="balanced", random_state=seed)

evaluate(rf)

## Predictions

In [ ]:
df_test = pd.read_csv(f"{root_path}/test_data.csv")
test_feats, _ = prep_df(df_test)
test_feats = scaler.transform(test_feats[features])

In [ ]:
subtask3 = rf.predict_proba(test_feats)[:, 1]

In [ ]:
plt.figure(figsize=(12,3))
sns.histplot(subtask3)

# Submission

In [ ]:
def build_submission(sid, ans):
    id_ = df_test["ID"] if sid == 3 else 1
    if sid == 2:
        id_ = [1, 2]
    return pd.DataFrame({"datapointID": id_, "subtaskID": sid, "answer": ans})

In [ ]:
subtasks = [
    (1, subtask1),
    (2, subtask2),
    (3, subtask3)
]

sub = pd.concat([build_submission(sid, ans) for sid, ans in subtasks])

In [ ]:
sub.head()

In [ ]:
sub.to_csv(f"{root_path}/submission.csv", index=False)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)